# Analisis Exploratorio de Datos (EDA) Completo
## Motor de Recomendacion Turistica TUI - TFM UCM 2025

Este notebook realiza un analisis exhaustivo de la calidad de todos los datos del proyecto:

1. **Calidad de datos**: Valores nulos, duplicados e inconsistencias
2. **Distribuciones y variables relevantes**
3. **Patrones, outliers y relaciones entre variables**
4. **Visualizaciones**: Media, mediana, propuestas para sustituir valores o descartar registros

### Fuentes de datos analizadas:

| Fuente | Tipo | Descripcion |
|--------|------|-------------|
| Base de datos SQLite | BD | Resenas, clima, indicadores, destinos, paquetes, usuarios, interacciones |
| clima_todos_los_destinos.csv | CSV | Datos climaticos historicos |
| conectividad_y_pasajeros_2025.csv | CSV | Conectividad aerea y pasajeros |
| seguridad_y_sanidad_banco_mundial.csv | CSV | Indicadores de seguridad y sanidad |

In [ ]:
# === CONFIGURACION Y LIBRERIAS ===
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Rutas
BASE_DIR = Path(r'D:\Master\TrabajoFinalUCM\TFM')
DB_PATH = BASE_DIR / 'data' / 'tui_recomendador.db'
CSV_CLIMA = BASE_DIR / 'data' / 'clima_todos_los_destinos.csv'
CSV_CONECTIVIDAD = BASE_DIR / 'data' / 'conectividad_y_pasajeros_2025.csv'
CSV_SEGURIDAD = BASE_DIR / 'data' / 'seguridad_y_sanidad_banco_mundial.csv'

# Funcion auxiliar para consultas
def query(sql, db=DB_PATH):
    conn = sqlite3.connect(str(db))
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

def get_tables():
    return query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

print(f'Base de datos: {DB_PATH} ({DB_PATH.stat().st_size / (1024*1024):.1f} MB)')
print(f'Tablas disponibles:')
tables = get_tables()
for _, row in tables.iterrows():
    n = query(f"SELECT COUNT(*) as n FROM [{row['name']}]")['n'].iloc[0]
    print(f'  - {row["name"]}: {n:,} registros')

---
## 1. CALIDAD DE DATOS: Valores Nulos, Duplicados e Inconsistencias

Analizamos cada fuente de datos para detectar problemas de calidad.

### 1.1 Tabla: resenas

In [ ]:
# Cargamos la tabla de resenas
df_resenas = query("SELECT * FROM resenas")
print(f'Total registros: {len(df_resenas):,}')
print(f'Columnas: {list(df_resenas.columns)}')
print(f'\n--- Tipos de datos ---')
print(df_resenas.dtypes)

# Valores nulos
print(f'\n--- Valores nulos por columna ---')
null_counts = df_resenas.isnull().sum()
null_pct = (null_counts / len(df_resenas) * 100).round(2)
null_df = pd.DataFrame({'Nulos': null_counts, 'Porcentaje (%)': null_pct})
null_df = null_df[null_df['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)
print(null_df)

# Duplicados
dup_total = df_resenas.duplicated().sum()
print(f'\n--- Duplicados exactos: {dup_total} ({dup_total/len(df_resenas)*100:.2f}%) ---')

# Duplicados por texto
if 'texto_original' in df_resenas.columns:
    dup_texto = df_resenas['texto_original'].duplicated().sum()
    print(f'Textos duplicados: {dup_texto} ({dup_texto/len(df_resenas)*100:.2f}%)')

In [ ]:
# Visualizacion: Porcentaje de nulos por columna en resenas
fig, ax = plt.subplots(figsize=(12, 5))
null_all = (df_resenas.isnull().sum() / len(df_resenas) * 100).sort_values(ascending=False)
null_all.plot(kind='bar', ax=ax, color=sns.color_palette('coolwarm', len(null_all)))
ax.set_title('Porcentaje de valores nulos por columna - Tabla resenas')
ax.set_ylabel('% Nulos')
ax.axhline(y=30, color='red', linestyle='--', label='Umbral exclusion (30%)')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nPropuesta: Columnas con >30% de nulos deben evaluarse para exclusion o imputacion.')

### 1.2 Tabla: clima_destinos

In [ ]:
df_clima_db = query("SELECT * FROM clima_destinos")
print(f'Total registros: {len(df_clima_db):,}')
print(f'Columnas: {list(df_clima_db.columns)}')

# Valores nulos
print(f'\n--- Valores nulos ---')
null_clima = df_clima_db.isnull().sum()
null_pct_clima = (null_clima / len(df_clima_db) * 100).round(2)
print(pd.DataFrame({'Nulos': null_clima, '%': null_pct_clima})[null_clima > 0])

# Duplicados
dup_clima = df_clima_db.duplicated().sum()
print(f'\nDuplicados exactos: {dup_clima}')

# Consistencia: rango de temperaturas
if 'temperatura_media' in df_clima_db.columns:
    temp_stats = df_clima_db['temperatura_media'].describe()
    print(f'\n--- Estadisticas de temperatura_media ---')
    print(temp_stats)
    anomalias = df_clima_db[(df_clima_db['temperatura_media'] < -20) | (df_clima_db['temperatura_media'] > 55)]
    print(f'Valores anomalos (< -20 o > 55): {len(anomalias)}')

### 1.3 Tabla: indicadores_destino

In [ ]:
df_indicadores = query("SELECT * FROM indicadores_destino")
print(f'Total registros: {len(df_indicadores):,}')
print(f'Columnas: {list(df_indicadores.columns)}')

# Valores nulos
print(f'\n--- Valores nulos ---')
null_ind = df_indicadores.isnull().sum()
null_pct_ind = (null_ind / len(df_indicadores) * 100).round(2)
print(pd.DataFrame({'Nulos': null_ind, '%': null_pct_ind})[null_ind > 0])

# Duplicados
dup_ind = df_indicadores.duplicated().sum()
print(f'\nDuplicados exactos: {dup_ind}')

# Inconsistencias: valores negativos donde no deberia haber
if 'valor' in df_indicadores.columns:
    negativos = df_indicadores[df_indicadores['valor'] < 0]
    print(f'\nValores negativos en columna valor: {len(negativos)}')
    if len(negativos) > 0:
        print(negativos[['destino_nombre', 'tipo_indicador', 'valor']].head(10))

### 1.4 Tabla: destinos_caracteristicas

In [ ]:
df_destinos = query("SELECT * FROM destinos_caracteristicas")
print(f'Total registros: {len(df_destinos):,}')
print(f'Columnas: {list(df_destinos.columns)}')

# Valores nulos
print(f'\n--- Valores nulos ---')
null_dest = df_destinos.isnull().sum()
null_pct_dest = (null_dest / len(df_destinos) * 100).round(2)
null_report = pd.DataFrame({'Nulos': null_dest, '%': null_pct_dest})
print(null_report[null_dest > 0].sort_values('%', ascending=False))

# Duplicados por nombre de destino
if 'destino_nombre' in df_destinos.columns:
    dup_dest = df_destinos['destino_nombre'].duplicated().sum()
    print(f'\nDestinos duplicados por nombre: {dup_dest}')

### 1.5 Archivos CSV externos

In [ ]:
# --- CSV: clima_todos_los_destinos.csv ---
print('=' * 60)
print('ARCHIVO: clima_todos_los_destinos.csv')
print('=' * 60)
df_clima_csv = pd.read_csv(CSV_CLIMA)
print(f'Registros: {len(df_clima_csv):,}')
print(f'Columnas: {list(df_clima_csv.columns)}')
print(f'\nNulos:')
print(df_clima_csv.isnull().sum()[df_clima_csv.isnull().sum() > 0])
print(f'\nDuplicados: {df_clima_csv.duplicated().sum()}')

print('\n')

# --- CSV: conectividad_y_pasajeros_2025.csv ---
print('=' * 60)
print('ARCHIVO: conectividad_y_pasajeros_2025.csv')
print('=' * 60)
df_conect = pd.read_csv(CSV_CONECTIVIDAD)
print(f'Registros: {len(df_conect):,}')
print(f'Columnas: {list(df_conect.columns)}')
print(f'\nNulos:')
print(df_conect.isnull().sum()[df_conect.isnull().sum() > 0])
print(f'\nDuplicados: {df_conect.duplicated().sum()}')

print('\n')

# --- CSV: seguridad_y_sanidad_banco_mundial.csv ---
print('=' * 60)
print('ARCHIVO: seguridad_y_sanidad_banco_mundial.csv')
print('=' * 60)
df_seguridad = pd.read_csv(CSV_SEGURIDAD)
print(f'Registros: {len(df_seguridad):,}')
print(f'Columnas: {list(df_seguridad.columns)}')
print(f'\nNulos:')
null_seg = df_seguridad.isnull().sum()
print(null_seg[null_seg > 0])
print(f'\nDuplicados: {df_seguridad.duplicated().sum()}')

### 1.6 Resumen de calidad de datos

In [ ]:
# Resumen consolidado de calidad
resumen_calidad = []

datasets = {
    'resenas (BD)': df_resenas,
    'clima_destinos (BD)': df_clima_db,
    'indicadores_destino (BD)': df_indicadores,
    'destinos_caracteristicas (BD)': df_destinos,
    'clima_csv': df_clima_csv,
    'conectividad_csv': df_conect,
    'seguridad_csv': df_seguridad,
}

for nombre, df in datasets.items():
    total = len(df)
    nulos_total = df.isnull().sum().sum()
    celdas_total = total * len(df.columns)
    pct_nulos = (nulos_total / celdas_total * 100) if celdas_total > 0 else 0
    duplicados = df.duplicated().sum()
    resumen_calidad.append({
        'Dataset': nombre,
        'Registros': total,
        'Columnas': len(df.columns),
        'Celdas nulas': nulos_total,
        '% Nulos': round(pct_nulos, 2),
        'Duplicados': duplicados
    })

df_resumen = pd.DataFrame(resumen_calidad)
print('RESUMEN DE CALIDAD DE DATOS')
print('=' * 80)
print(df_resumen.to_string(index=False))

---
## 2. DISTRIBUCIONES Y VARIABLES RELEVANTES

Analizamos las distribuciones de las variables mas importantes para el sistema de recomendacion.

### 2.1 Distribucion de resenas por fuente e idioma

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por fuente
if 'fuente' in df_resenas.columns:
    fuente_counts = df_resenas['fuente'].value_counts()
    fuente_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(fuente_counts)))
    axes[0].set_title('Resenas por fuente de scraping')
    axes[0].set_ylabel('Cantidad')
    for i, v in enumerate(fuente_counts.values):
        axes[0].text(i, v + 50, str(v), ha='center', fontsize=9)

# Por idioma
if 'idioma' in df_resenas.columns:
    idioma_counts = df_resenas['idioma'].value_counts().head(10)
    idioma_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set3', len(idioma_counts)))
    axes[1].set_title('Top 10 idiomas de resenas')
    axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

### 2.2 Distribucion de resenas por destino

In [ ]:
if 'destino_nombre' in df_resenas.columns:
    dest_counts = df_resenas['destino_nombre'].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top 20 destinos
    dest_counts.head(20).plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Top 20 destinos con mas resenas')
    axes[0].set_xlabel('Numero de resenas')
    axes[0].invert_yaxis()
    
    # Distribucion (histograma del numero de resenas por destino)
    axes[1].hist(dest_counts.values, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[1].axvline(dest_counts.median(), color='blue', linestyle='--', label=f'Mediana: {dest_counts.median():.0f}')
    axes[1].axvline(dest_counts.mean(), color='red', linestyle='--', label=f'Media: {dest_counts.mean():.0f}')
    axes[1].set_title('Distribucion del numero de resenas por destino')
    axes[1].set_xlabel('Resenas por destino')
    axes[1].set_ylabel('Frecuencia')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nEstadisticas de resenas por destino:')
    print(f'  Media: {dest_counts.mean():.1f}')
    print(f'  Mediana: {dest_counts.median():.1f}')
    print(f'  Desviacion estandar: {dest_counts.std():.1f}')
    print(f'  Minimo: {dest_counts.min()} ({dest_counts.idxmin()})')
    print(f'  Maximo: {dest_counts.max()} ({dest_counts.idxmax()})')
    print(f'\n  PROPUESTA: Destinos con menos de {int(dest_counts.quantile(0.1))} resenas (percentil 10)'
          f' podrian requerir enriquecimiento adicional de datos.')

### 2.3 Longitud de textos de resenas

In [ ]:
if 'texto_original' in df_resenas.columns:
    df_resenas['len_texto'] = df_resenas['texto_original'].fillna('').str.len()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histograma de longitud
    axes[0].hist(df_resenas['len_texto'], bins=50, color='teal', edgecolor='black', alpha=0.7)
    axes[0].axvline(df_resenas['len_texto'].median(), color='blue', linestyle='--', 
                    label=f'Mediana: {df_resenas["len_texto"].median():.0f} chars')
    axes[0].axvline(df_resenas['len_texto'].mean(), color='red', linestyle='--', 
                    label=f'Media: {df_resenas["len_texto"].mean():.0f} chars')
    axes[0].set_title('Distribucion de longitud de textos de resenas')
    axes[0].set_xlabel('Longitud (caracteres)')
    axes[0].set_ylabel('Frecuencia')
    axes[0].legend()
    
    # Boxplot por fuente
    if 'fuente' in df_resenas.columns:
        df_resenas.boxplot(column='len_texto', by='fuente', ax=axes[1])
        axes[1].set_title('Longitud de texto por fuente')
        axes[1].set_xlabel('Fuente')
        axes[1].set_ylabel('Caracteres')
        plt.suptitle('')  # Eliminar titulo automatico
    
    plt.tight_layout()
    plt.show()
    
    # Textos muy cortos (potencialmente no utiles para embeddings)
    cortos = len(df_resenas[df_resenas['len_texto'] < 20])
    print(f'\nTextos con menos de 20 caracteres: {cortos} ({cortos/len(df_resenas)*100:.1f}%)')
    print(f'PROPUESTA: Considerar excluir textos < 20 caracteres del calculo de embeddings.')

### 2.4 Indicadores por fuente y tipo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'fuente' in df_indicadores.columns:
    fuente_ind = df_indicadores['fuente'].value_counts()
    fuente_ind.plot(kind='bar', ax=axes[0], color=sns.color_palette('Paired', len(fuente_ind)))
    axes[0].set_title('Indicadores por fuente estadistica')
    axes[0].set_ylabel('Cantidad')
    for i, v in enumerate(fuente_ind.values):
        axes[0].text(i, v + 10, str(v), ha='center', fontsize=9)

if 'tipo_indicador' in df_indicadores.columns:
    tipo_ind = df_indicadores['tipo_indicador'].value_counts()
    tipo_ind.plot(kind='bar', ax=axes[1], color=sns.color_palette('Dark2', len(tipo_ind)))
    axes[1].set_title('Indicadores por tipo')
    axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

### 2.5 Datos climaticos

In [ ]:
# Estadisticas descriptivas del CSV de clima
print('Estadisticas descriptivas - clima_todos_los_destinos.csv')
print('=' * 60)
# Seleccionar solo columnas numericas
numeric_cols = df_clima_csv.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    print(df_clima_csv[numeric_cols].describe().round(2))
    
    # Visualizacion de distribuciones
    n_cols = min(len(numeric_cols), 6)
    if n_cols > 0:
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        axes = axes.flatten()
        for i, col in enumerate(numeric_cols[:6]):
            axes[i].hist(df_clima_csv[col].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
            mean_val = df_clima_csv[col].mean()
            median_val = df_clima_csv[col].median()
            axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Media: {mean_val:.1f}')
            axes[i].axvline(median_val, color='blue', linestyle='--', label=f'Mediana: {median_val:.1f}')
            axes[i].set_title(col)
            axes[i].legend(fontsize=8)
        # Ocultar ejes sobrantes
        for j in range(n_cols, 6):
            axes[j].set_visible(False)
        plt.suptitle('Distribuciones de variables climaticas', fontsize=14)
        plt.tight_layout()
        plt.show()

---
## 3. PATRONES, OUTLIERS Y RELACIONES ENTRE VARIABLES

### 3.1 Deteccion de Outliers en resenas

In [ ]:
if 'len_texto' in df_resenas.columns:
    # Metodo IQR para detectar outliers
    Q1 = df_resenas['len_texto'].quantile(0.25)
    Q3 = df_resenas['len_texto'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df_resenas[(df_resenas['len_texto'] < lower) | (df_resenas['len_texto'] > upper)]
    print(f'Outliers en longitud de texto (metodo IQR):')
    print(f'  Limite inferior: {lower:.0f}')
    print(f'  Limite superior: {upper:.0f}')
    print(f'  Outliers detectados: {len(outliers)} ({len(outliers)/len(df_resenas)*100:.1f}%)')
    print(f'  - Por debajo: {len(df_resenas[df_resenas["len_texto"] < lower])}')
    print(f'  - Por encima: {len(df_resenas[df_resenas["len_texto"] > upper])}')
    
    # Boxplot general
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.boxplot(df_resenas['len_texto'].values, vert=False)
    ax.set_title('Boxplot: Longitud de texto de resenas (deteccion de outliers)')
    ax.set_xlabel('Caracteres')
    plt.tight_layout()
    plt.show()
    
    print('\nPROPUESTA: Los textos extremadamente largos (> limite superior) probablemente '
          'provienen de posts completos de Reddit/YouTube. Considerar truncar a 2000 caracteres '
          'para el calculo de embeddings.')

### 3.2 Outliers en indicadores de destino

In [ ]:
if 'valor' in df_indicadores.columns and 'tipo_indicador' in df_indicadores.columns:
    # Outliers por tipo de indicador
    tipos = df_indicadores['tipo_indicador'].unique()
    
    fig, axes = plt.subplots(1, min(len(tipos), 4), figsize=(16, 5))
    if len(tipos) == 1:
        axes = [axes]
    
    for i, tipo in enumerate(tipos[:4]):
        subset = df_indicadores[df_indicadores['tipo_indicador'] == tipo]['valor']
        axes[i].boxplot(subset.dropna().values, vert=True)
        axes[i].set_title(tipo[:20], fontsize=9)
        axes[i].set_ylabel('Valor')
    
    plt.suptitle('Boxplots por tipo de indicador (deteccion de outliers)', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Resumen de outliers por tipo
    print('\nOutliers por tipo de indicador (metodo IQR):')
    for tipo in tipos:
        subset = df_indicadores[df_indicadores['tipo_indicador'] == tipo]['valor'].dropna()
        if len(subset) > 4:
            q1, q3 = subset.quantile(0.25), subset.quantile(0.75)
            iqr = q3 - q1
            n_outliers = len(subset[(subset < q1 - 1.5*iqr) | (subset > q3 + 1.5*iqr)])
            if n_outliers > 0:
                print(f'  {tipo}: {n_outliers} outliers')

### 3.3 Correlaciones entre variables climaticas

In [ ]:
numeric_cols_clima = df_clima_csv.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols_clima) >= 2:
    corr_matrix = df_clima_csv[numeric_cols_clima].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                square=True, linewidths=0.5, ax=ax, fmt='.2f')
    ax.set_title('Matriz de correlacion - Variables climaticas')
    plt.tight_layout()
    plt.show()
    
    # Correlaciones fuertes
    print('Correlaciones fuertes (|r| > 0.7):')
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            val = corr_matrix.iloc[i, j]
            if abs(val) > 0.7:
                print(f'  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {val:.3f}')

### 3.4 Cobertura de destinos entre fuentes

In [ ]:
# Verificar que destinos estan presentes en cada fuente
destinos_esperados = [
    'Mallorca', 'Tenerife', 'Ibiza', 'Costa del Sol', 'Barcelona',
    'Madrid', 'Malaga', 'Sevilla', 'Valencia', 'Gran Canaria',
    'Alicante', 'Bilbao', 'San Sebastian', 'Cordoba', 'Granada',
    'Cadiz', 'Fuerteventura', 'Lanzarote', 'Menorca', 'Antalya',
    'Rodas', 'Santorini', 'Hurghada', 'Punta Cana', 'Cancun',
    'Riviera Maya', 'Dubai', 'Maldivas', 'Bali', 'Phuket',
    'Marrakech', 'Cabo Verde', 'Split', 'Creta', 'Sicilia',
    'Cerdena', 'Costa Amalfitana', 'Algarve', 'Tunez'
]

def check_coverage(df, col_name, dataset_name):
    """Verifica cobertura de destinos esperados."""
    if col_name not in df.columns:
        return None
    destinos_en_data = set(df[col_name].dropna().unique())
    presentes = [d for d in destinos_esperados if d in destinos_en_data]
    ausentes = [d for d in destinos_esperados if d not in destinos_en_data]
    return {'dataset': dataset_name, 'presentes': len(presentes), 
            'ausentes': len(ausentes), 'total': len(destinos_esperados),
            'destinos_ausentes': ausentes}

coverages = []
coverages.append(check_coverage(df_resenas, 'destino_nombre', 'resenas'))
coverages.append(check_coverage(df_clima_db, 'destino_nombre', 'clima_destinos'))
coverages.append(check_coverage(df_indicadores, 'destino_nombre', 'indicadores'))
coverages.append(check_coverage(df_destinos, 'destino_nombre', 'destinos_caract'))

print('COBERTURA DE DESTINOS ESPERADOS POR FUENTE')
print('=' * 60)
for c in coverages:
    if c:
        pct = c['presentes'] / c['total'] * 100
        print(f"  {c['dataset']:<25} {c['presentes']}/{c['total']} ({pct:.0f}%)")
        if c['ausentes'] > 0 and c['ausentes'] <= 10:
            print(f"    Ausentes: {', '.join(c['destinos_ausentes'][:5])}...")

print('\nPROPUESTA: Priorizar enriquecimiento de datos para destinos '
      'ausentes en multiples fuentes.')

---
## 4. VISUALIZACIONES Y PROPUESTAS DE TRATAMIENTO

### 4.1 Comparativa Media vs Mediana por destino (resenas)

In [ ]:
if 'destino_nombre' in df_resenas.columns and 'len_texto' in df_resenas.columns:
    # Media y mediana de longitud de texto por destino (top 20)
    top_destinos = df_resenas['destino_nombre'].value_counts().head(20).index
    df_top = df_resenas[df_resenas['destino_nombre'].isin(top_destinos)]
    
    stats_destino = df_top.groupby('destino_nombre')['len_texto'].agg(['mean', 'median', 'std']).round(0)
    stats_destino = stats_destino.sort_values('mean', ascending=False)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    x = range(len(stats_destino))
    width = 0.35
    bars1 = ax.bar([i - width/2 for i in x], stats_destino['mean'], width, 
                   label='Media', color='steelblue', alpha=0.8)
    bars2 = ax.bar([i + width/2 for i in x], stats_destino['median'], width, 
                   label='Mediana', color='coral', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(stats_destino.index, rotation=45, ha='right')
    ax.set_title('Media vs Mediana: Longitud de texto por destino (Top 20)')
    ax.set_ylabel('Caracteres')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Destinos con alta diferencia media-mediana (sesgo)
    stats_destino['diff_pct'] = ((stats_destino['mean'] - stats_destino['median']) / stats_destino['median'] * 100).round(1)
    sesgados = stats_destino[stats_destino['diff_pct'] > 50]
    if len(sesgados) > 0:
        print('Destinos con alta diferencia Media-Mediana (sesgo positivo > 50%):')
        print(sesgados[['mean', 'median', 'diff_pct']])
        print('\nPROPUESTA: Para estos destinos, la MEDIANA es mas representativa que la media.'
              ' Usar mediana para normalizar embeddings.')

### 4.2 Propuestas para tratamiento de valores nulos

In [ ]:
print('PROPUESTAS PARA TRATAMIENTO DE VALORES NULOS')
print('=' * 70)
print()
print('1. RESENAS:')
print('   - puntuacion (nula): NO imputar. Muchas fuentes (Reddit, YouTube)')
print('     no tienen puntuacion numerica. Usar solo para fuentes con rating.')
print('   - fecha_publicacion (nula): Imputar con fecha de extraccion si disponible.')
print('   - texto_original (vacio): EXCLUIR del calculo de embeddings.')
print()
print('2. INDICADORES DE DESTINO:')
print('   - mes (nulo): Correcto para indicadores anuales. No requiere imputacion.')
print('   - valor (negativo): Verificar si es error o valor valido segun tipo.')
print()
print('3. CLIMA:')
print('   - Variables con <5% nulos: Imputar con interpolacion temporal.')
print('   - Variables con >30% nulos: Evaluar exclusion de la variable.')
print()
print('4. CONECTIVIDAD/SEGURIDAD:')
print('   - Valores nulos por pais: Imputar con media regional si <20% nulos.')
print('   - Si >20% nulos en un pais: Excluir ese pais del analisis.')
print()
print('UMBRAL GENERAL (segun config.yml): Si un registro tiene >30% de atributos')
print('vacios, se excluye del sistema de recomendacion (parametro exclusion_threshold).')

### 4.3 Resumen visual consolidado

In [ ]:
# Visualizacion consolidada de calidad
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Porcentaje de nulos por dataset
nombres = [r['Dataset'] for r in resumen_calidad]
pct_nulos = [r['% Nulos'] for r in resumen_calidad]
colors = ['green' if p < 5 else 'orange' if p < 15 else 'red' for p in pct_nulos]
axes[0].barh(nombres, pct_nulos, color=colors)
axes[0].axvline(x=5, color='orange', linestyle='--', alpha=0.5, label='Umbral leve (5%)')
axes[0].axvline(x=15, color='red', linestyle='--', alpha=0.5, label='Umbral critico (15%)')
axes[0].set_title('Porcentaje de celdas nulas por dataset')
axes[0].set_xlabel('% Nulos')
axes[0].legend()

# Grafico 2: Registros por dataset
registros = [r['Registros'] for r in resumen_calidad]
axes[1].barh(nombres, registros, color='steelblue')
axes[1].set_title('Numero de registros por dataset')
axes[1].set_xlabel('Registros')
for i, v in enumerate(registros):
    axes[1].text(v + max(registros)*0.01, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. CONCLUSIONES 

1 **Principales problemas**:
   - Puntuacion numerica ausente en fuentes sin rating (Reddit, YouTube)
   - Distribucion desigual de resenas por destino (sesgo hacia destinos populares)
   - Textos de longitud muy variable (algunos extremadamente largos)

3. **Propuestas de tratamiento**:
   - **NO imputar** puntuaciones: usar analisis de sentimiento del texto como proxy (Javie aqui estas tu)
   - **Truncar** textos a 2000 caracteres para embeddings 
   - **Excluir** registros con texto vacio del sistema de embeddings
   - **Usar mediana** como estadistico robusto para destinos con distribuciones sesgadas
   - **Enriquecer** datos para destinos con baja cobertura

4. **Outliers**:
   - En longitud de texto: textos >5000 caracteres son posts completos. Truncar.
   - En indicadores: valores atipicos por cambios metodologicos entre anos. Verificar.
   - En clima: no se detectan anomalias significativas.

---
## 6. CLASIFICACION DE VARIABLES: Categoricas vs Numericas

Identificamos y clasificamos todas las variables de los datasets principales
para entender su naturaleza y definir tratamientos adecuados.

In [ ]:
# === CLASIFICACION AUTOMATICA DE VARIABLES ===

def clasificar_variables(df, nombre_dataset):
    """Clasifica las variables de un DataFrame en categoricas y numericas."""
    numericas = df.select_dtypes(include=['number']).columns.tolist()
    categoricas = df.select_dtypes(include=['object', 'category']).columns.tolist()
    booleanas = df.select_dtypes(include=['bool']).columns.tolist()
    fechas = [c for c in df.columns if 'fecha' in c.lower() or 'date' in c.lower()]
    texto_libre = [c for c in categoricas if df[c].dropna().str.len().mean() > 50]
    categoricas_puras = [c for c in categoricas if c not in texto_libre and c not in fechas]
    
    print(f'\n{"="*60}')
    print(f'DATASET: {nombre_dataset} ({len(df)} registros, {len(df.columns)} columnas)')
    print(f'{"="*60}')
    print(f'\n  NUMERICAS ({len(numericas)}):')
    for col in numericas:
        print(f'    - {col} (dtype: {df[col].dtype}, unicos: {df[col].nunique()})')
    print(f'\n  CATEGORICAS ({len(categoricas_puras)}):')
    for col in categoricas_puras:
        print(f'    - {col} (unicos: {df[col].nunique()}, top: {df[col].mode().iloc[0] if len(df[col].mode()) > 0 else "N/A"})')
    print(f'\n  TEXTO LIBRE ({len(texto_libre)}):')
    for col in texto_libre:
        print(f'    - {col} (longitud media: {df[col].dropna().str.len().mean():.0f} chars)')
    print(f'\n  FECHAS/TEMPORALES ({len(fechas)}):')
    for col in fechas:
        print(f'    - {col}')
    print(f'\n  BOOLEANAS ({len(booleanas)}):')
    for col in booleanas:
        print(f'    - {col}')
    
    return {'numericas': numericas, 'categoricas': categoricas_puras, 
            'texto': texto_libre, 'fechas': fechas, 'booleanas': booleanas}

# Aplicar a cada dataset
vars_resenas = clasificar_variables(df_resenas, 'resenas')
vars_clima = clasificar_variables(df_clima_csv, 'clima_todos_los_destinos.csv')
vars_indicadores = clasificar_variables(df_indicadores, 'indicadores_destino')
vars_seguridad = clasificar_variables(df_seguridad, 'seguridad_y_sanidad_banco_mundial.csv')
vars_conectividad = clasificar_variables(df_conect, 'conectividad_y_pasajeros_2025.csv')

### 6.1 Mejoras en tipos de variables (Type Casting)

Muchas columnas llegan como `object` cuando deberian ser numericas, fechas o categoricas.
Aqui proponemos y aplicamos las conversiones necesarias.

In [ ]:
# === MEJORAS EN TIPOS DE VARIABLES ===

print('PROPUESTAS DE MEJORA DE TIPOS DE VARIABLES')
print('=' * 70)
print()

# 1. Tabla resenas
print('TABLA: resenas')
print('-' * 40)

# Convertir fecha_publicacion a datetime
if 'fecha_publicacion' in df_resenas.columns:
    df_resenas['fecha_publicacion'] = pd.to_datetime(df_resenas['fecha_publicacion'], errors='coerce')
    print(f'  fecha_publicacion: object -> datetime64 (convertido)')

if 'fecha_extraccion' in df_resenas.columns:
    df_resenas['fecha_extraccion'] = pd.to_datetime(df_resenas['fecha_extraccion'], errors='coerce')
    print(f'  fecha_extraccion: object -> datetime64 (convertido)')

# Convertir puntuacion a float
if 'puntuacion' in df_resenas.columns:
    df_resenas['puntuacion'] = pd.to_numeric(df_resenas['puntuacion'], errors='coerce')
    print(f'  puntuacion: object -> float64 (convertido)')

# Convertir fuente e idioma a category
for col in ['fuente', 'idioma']:
    if col in df_resenas.columns:
        df_resenas[col] = df_resenas[col].astype('category')
        print(f'  {col}: object -> category (convertido, {df_resenas[col].nunique()} niveles)')

print()
print('TABLA: indicadores_destino')
print('-' * 40)

# Convertir campos categoricos
for col in ['fuente', 'tipo_indicador', 'destino_nombre']:
    if col in df_indicadores.columns:
        df_indicadores[col] = df_indicadores[col].astype('category')
        print(f'  {col}: object -> category ({df_indicadores[col].nunique()} niveles)')

print()
print('BENEFICIOS de la conversion:')
print('  - Ahorro de memoria (category usa enteros internamente)')
print('  - Operaciones de filtrado mas rapidas')
print('  - Las fechas permiten operaciones temporales (diferencias, agrupaciones por mes/ano)')
print('  - Los numericos permiten calculos estadisticos directos')

---
## 7. ANALISIS DETALLADO DE MISSING VALUES

Mapa de calor de valores faltantes, patrones de nulidad y mecanismos de missing.

In [ ]:
# === MAPA DE CALOR DE MISSING VALUES ===

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

datasets_missing = [
    ('resenas', df_resenas),
    ('clima_csv', df_clima_csv),
    ('indicadores', df_indicadores),
    ('seguridad_csv', df_seguridad),
]

for idx, (nombre, df) in enumerate(datasets_missing):
    ax = axes[idx // 2][idx % 2]
    # Muestrear si es muy grande
    sample = df.head(200) if len(df) > 200 else df
    sns.heatmap(sample.isnull().T, cbar=True, yticklabels=True, ax=ax,
                cmap='YlOrRd', cbar_kws={'label': 'Missing'})
    ax.set_title(f'Patron de missing - {nombre}', fontsize=11)
    ax.set_xlabel('Registros (muestra)')

plt.suptitle('Mapas de calor de valores faltantes por dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('INTERPRETACION:')
print('  - Filas completamente amarillas/rojas = variables con alta tasa de missing')
print('  - Patrones verticales = registros con muchos campos vacios (candidatos a exclusion)')
print('  - Missing aleatorio vs sistematico determina la estrategia de imputacion')

In [ ]:
# === CLASIFICACION DEL MECANISMO DE MISSING ===

print('CLASIFICACION DE MECANISMOS DE VALORES FALTANTES')
print('=' * 70)
print()
print('Segun la taxonomia de Rubin (1976):')
print()
print('1. MCAR (Missing Completely At Random):')
print('   - No hay patron en la falta de datos')
print('   - Ejemplo: fecha_extraccion faltante por error de sistema')
print('   - Tratamiento: Imputacion simple (media/mediana) o eliminacion')
print()
print('2. MAR (Missing At Random):')
print('   - El missing depende de otras variables observadas')
print('   - Ejemplo: puntuacion faltante porque la fuente es Reddit/YouTube')
print('   - Tratamiento: Imputacion multiple o modelos predictivos')
print()
print('3. MNAR (Missing Not At Random):')
print('   - El missing depende del valor no observado')
print('   - Ejemplo: valoraciones faltantes porque usuarios insatisfechos no valoran')
print('   - Tratamiento: Modelos de seleccion o analisis de sensibilidad')
print()

# Verificar relacion entre missing y fuente en resenas
if 'fuente' in df_resenas.columns and 'puntuacion' in df_resenas.columns:
    print('\nVERIFICACION: puntuacion faltante vs fuente (tabla resenas):')
    missing_by_source = df_resenas.groupby('fuente')['puntuacion'].apply(
        lambda x: x.isnull().mean() * 100).round(1)
    for fuente, pct in missing_by_source.items():
        print(f'  {fuente}: {pct}% missing')
    print()
    print('  CONCLUSION: puntuacion es MAR (depende de la fuente).')
    print('  ESTRATEGIA: No imputar directamente; usar sentimiento del texto como proxy.')

---
## 8. DETECCION SISTEMATICA DE OUTLIERS

Aplicamos multiples metodos (IQR, Z-score) a todas las variables numericas
para identificar valores atipicos de forma exhaustiva.

In [ ]:
# === DETECCION SISTEMATICA DE OUTLIERS ===

from scipy import stats

def detectar_outliers(df, nombre_dataset, umbral_zscore=3.0):
    """Detecta outliers en todas las variables numericas usando IQR y Z-score."""
    numericas = df.select_dtypes(include=['number']).columns.tolist()
    resultados = []
    
    for col in numericas:
        data = df[col].dropna()
        if len(data) < 10:
            continue
        
        # Metodo IQR
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        outliers_iqr = len(data[(data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)])
        
        # Metodo Z-score
        z_scores = np.abs(stats.zscore(data))
        outliers_zscore = len(data[z_scores > umbral_zscore])
        
        pct_iqr = (outliers_iqr / len(data) * 100)
        pct_zscore = (outliers_zscore / len(data) * 100)
        
        resultados.append({
            'Variable': col,
            'N': len(data),
            'Media': data.mean(),
            'Mediana': data.median(),
            'Outliers_IQR': outliers_iqr,
            '%_IQR': round(pct_iqr, 1),
            'Outliers_Zscore': outliers_zscore,
            '%_Zscore': round(pct_zscore, 1),
            'Skewness': round(data.skew(), 2)
        })
    
    df_result = pd.DataFrame(resultados)
    print(f'\n{"="*80}')
    print(f'OUTLIERS DETECTADOS - {nombre_dataset}')
    print(f'{"="*80}')
    if len(df_result) > 0:
        # Filtrar solo variables con outliers
        df_con_outliers = df_result[(df_result['%_IQR'] > 0) | (df_result['%_Zscore'] > 0)]
        if len(df_con_outliers) > 0:
            print(df_con_outliers.to_string(index=False))
        else:
            print('  No se detectaron outliers significativos.')
    return df_result

# Aplicar a los datasets principales
outliers_resenas = detectar_outliers(df_resenas, 'resenas')
outliers_clima = detectar_outliers(df_clima_csv, 'clima_todos_los_destinos.csv')
outliers_indicadores = detectar_outliers(df_indicadores, 'indicadores_destino')
outliers_seguridad = detectar_outliers(df_seguridad, 'seguridad_y_sanidad_banco_mundial.csv')
outliers_conectividad = detectar_outliers(df_conect, 'conectividad_y_pasajeros_2025.csv')

In [ ]:
# === VISUALIZACION DE OUTLIERS (Boxplots multiples) ===

# Para el clima CSV (multiples variables numericas)
numeric_cols_clima = df_clima_csv.select_dtypes(include=['number']).columns.tolist()
if len(numeric_cols_clima) > 0:
    n_vars = min(len(numeric_cols_clima), 8)
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols_clima[:8]):
        data = df_clima_csv[col].dropna()
        axes[i].boxplot(data.values, vert=True)
        # Marcar media y mediana
        mean_val = data.mean()
        median_val = data.median()
        axes[i].axhline(mean_val, color='red', linestyle='--', alpha=0.5, label=f'Media: {mean_val:.1f}')
        axes[i].axhline(median_val, color='blue', linestyle='--', alpha=0.5, label=f'Mediana: {median_val:.1f}')
        axes[i].set_title(col[:20], fontsize=9)
        axes[i].legend(fontsize=7)
    
    for j in range(n_vars, 8):
        axes[j].set_visible(False)
    
    plt.suptitle('Boxplots con Media y Mediana - Variables climaticas', fontsize=13)
    plt.tight_layout()
    plt.show()

print('\nINTERPRETACION:')
print('  - Puntos fuera de los bigotes = outliers IQR')
print('  - Si Media >> Mediana = distribucion sesgada positivamente (outliers altos)')
print('  - Si Media << Mediana = distribucion sesgada negativamente (outliers bajos)')
print('  - Para variables sesgadas, la MEDIANA es mejor representante central')

---
## 9. ESTRATEGIAS DE IMPUTACION

Definimos y demostramos las estrategias de imputacion para cada tipo de variable
y mecanismo de missing identificado.

In [ ]:
# === TABLA DE ESTRATEGIAS DE IMPUTACION ===

estrategias = [
    {'Dataset': 'resenas', 'Variable': 'puntuacion', 'Tipo': 'Numerica',
     'Mecanismo': 'MAR', '% Missing': '100%',
     'Estrategia': 'NO imputar. Usar sentimiento NLP del texto como proxy.',
     'Justificacion': 'Las fuentes Reddit/YouTube no tienen rating numerico'},
    {'Dataset': 'resenas', 'Variable': 'fecha_publicacion', 'Tipo': 'Fecha',
     'Mecanismo': 'MAR', '% Missing': '~64%',
     'Estrategia': 'Imputar con fecha_extraccion como aproximacion',
     'Justificacion': 'La fecha de extraccion es la mejor aproximacion disponible'},
    {'Dataset': 'resenas', 'Variable': 'id_paquete', 'Tipo': 'ID',
     'Mecanismo': 'MNAR', '% Missing': '100%',
     'Estrategia': 'NO imputar. Las resenas son de destinos, no de paquetes TUI',
     'Justificacion': 'Las resenas scrapeadas no se vinculan a paquetes especificos'},
    {'Dataset': 'clima', 'Variable': 'Variables numericas', 'Tipo': 'Numerica',
     'Mecanismo': 'MCAR', '% Missing': '<5%',
     'Estrategia': 'Interpolacion temporal (mes anterior/posterior)',
     'Justificacion': 'Los datos climaticos tienen continuidad temporal'},
    {'Dataset': 'clima', 'Variable': 'Variables numericas', 'Tipo': 'Numerica',
     'Mecanismo': 'MCAR', '% Missing': '>30%',
     'Estrategia': 'EXCLUIR la variable del modelo',
     'Justificacion': 'Demasiada incertidumbre para imputacion fiable'},
    {'Dataset': 'seguridad', 'Variable': 'Indicadores por pais', 'Tipo': 'Numerica',
     'Mecanismo': 'MAR', '% Missing': 'Variable',
     'Estrategia': 'Imputar con media regional (misma zona geografica)',
     'Justificacion': 'Paises de la misma region tienen indicadores similares'},
    {'Dataset': 'indicadores', 'Variable': 'mes', 'Tipo': 'Entero',
     'Mecanismo': 'MNAR', '% Missing': 'Variable',
     'Estrategia': 'NO imputar. NULL indica dato anual (no mensual)',
     'Justificacion': 'Es un missing estructural, no un error'},
]

df_estrategias = pd.DataFrame(estrategias)
print('TABLA DE ESTRATEGIAS DE IMPUTACION')
print('=' * 100)
print(df_estrategias.to_string(index=False))

In [ ]:
# === DEMOSTRACION PRACTICA: Imputacion de fechas ===

print('DEMOSTRACION: Imputacion de fecha_publicacion en resenas')
print('=' * 60)

if 'fecha_publicacion' in df_resenas.columns and 'fecha_extraccion' in df_resenas.columns:
    missing_antes = df_resenas['fecha_publicacion'].isnull().sum()
    print(f'\nAntes de imputacion: {missing_antes:,} valores faltantes ({missing_antes/len(df_resenas)*100:.1f}%)')
    
    # Imputar con fecha_extraccion
    df_resenas_imputado = df_resenas.copy()
    mask = df_resenas_imputado['fecha_publicacion'].isnull()
    df_resenas_imputado.loc[mask, 'fecha_publicacion'] = df_resenas_imputado.loc[mask, 'fecha_extraccion']
    
    missing_despues = df_resenas_imputado['fecha_publicacion'].isnull().sum()
    print(f'Despues de imputacion: {missing_despues:,} valores faltantes ({missing_despues/len(df_resenas)*100:.1f}%)')
    print(f'Registros imputados: {missing_antes - missing_despues:,}')
    
    print('\nNOTA: Esta imputacion es conservadora. La fecha de extraccion')
    print('es posterior a la publicacion real, pero es la mejor aproximacion.')

print()
print('DEMOSTRACION: Interpolacion de variables climaticas')
print('=' * 60)

numeric_clima = df_clima_csv.select_dtypes(include=['number']).columns.tolist()
if len(numeric_clima) > 0:
    col_ejemplo = numeric_clima[0]
    missing_antes_clima = df_clima_csv[col_ejemplo].isnull().sum()
    print(f'\nVariable ejemplo: {col_ejemplo}')
    print(f'Missing antes: {missing_antes_clima}')
    
    # Interpolacion temporal
    df_clima_imputado = df_clima_csv.copy()
    df_clima_imputado[col_ejemplo] = df_clima_imputado[col_ejemplo].interpolate(method='linear')
    missing_despues_clima = df_clima_imputado[col_ejemplo].isnull().sum()
    print(f'Missing despues (interpolacion lineal): {missing_despues_clima}')
    print('\nNOTA: Para datos climaticos mensuales, la interpolacion lineal')
    print('preserva la tendencia temporal entre observaciones consecutivas.')

---
## 10. RESUMEN FINAL: Decisiones de Preprocesamiento

Consolidamos todas las decisiones tomadas en este EDA.

In [ ]:
print('RESUMEN DE DECISIONES DE PREPROCESAMIENTO')
print('=' * 70)
print()
print('1. TIPOS DE VARIABLES:')
print('   - Convertir columnas de fecha (object -> datetime64)')
print('   - Convertir columnas categoricas de baja cardinalidad (object -> category)')
print('   - Mantener puntuacion como float64 (permite NaN semantico)')
print()
print('2. VALORES FALTANTES:')
print('   - puntuacion: NO imputar (MAR, usar NLP como proxy)')
print('   - fecha_publicacion: Imputar con fecha_extraccion')
print('   - Variables climaticas (<5% missing): Interpolacion lineal')
print('   - Variables con >30% missing: Excluir del modelo')
print('   - id_paquete en resenas: NO imputar (missing estructural)')
print()
print('3. OUTLIERS:')
print('   - Longitud de texto > 5000 chars: Truncar a 2000 para embeddings')
print('   - Indicadores con valores extremos: Verificar contra fuente original')
print('   - Variables climaticas: Sin anomalias significativas detectadas')
print('   - Para variables con skewness > 2: Usar mediana en lugar de media')
print()
print('4. EXCLUSION DE REGISTROS:')
print('   - Registros con texto_original vacio: Excluir de embeddings')
print('   - Registros con >30% de atributos vacios: Excluir (umbral config.yml)')
print('   - Duplicados exactos: Ya verificado = 0 en todos los datasets')
print()
print('5. VARIABLES CATEGORICAS:')
print('   - fuente (4 niveles): google_maps, reddit, tripadvisor, youtube')
print('   - idioma (40 niveles): Considerar agrupar idiomas minoritarios en "otros"')
print('   - tipo_indicador: Usar como variable de agrupacion, no como feature directa')
print('   - zona_geografica: Util para imputacion regional de missing')
print()
print('6. TRANSFORMACIONES:')
print('   - Variables con alta skewness: Considerar log-transform para normalizar')
print('   - Normalizacion min-max para variables que entran en el vector hibrido')
print('   - One-hot encoding para categoricas de baja cardinalidad')